# [3단계 GeoAI] 제로샷 전력설비 탐지 — Grounding DINO

국토위성 0.5m 타일(파일럿 3개 권역)에서 **철탑(tower)·변전소(substation)** 를
라벨 없이 탐지하고, 타일 매니페스트의 아핀으로 탐지 중심을 EPSG:5179 좌표로
변환해 CSV로 저장한다. OSM 대조(정밀도·재현율)는 로컬 파이프라인이 수행한다.

**Drive를 쓰지 않는다** (기관망에서 Drive 차단). 타일 반입 경로는 두 가지:

- **경로 A (전체 실행, 권장)** — HuggingFace 비공개 데이터셋. 로컬 PC에서
  `scripts/s3_hf_transfer.py upload` 로 타일 zip을 올려두고, 이 노트북이
  토큰으로 내려받는다. 결과도 같은 데이터셋에 되올리면 로컬에서
  `scripts/s3_hf_transfer.py download` 로 회수한다.
- **경로 B (장면 1~2개 시험)** — Colab 좌측 파일탭에 장면 zip과
  `tiles_manifest.csv` 를 직접 끌어다 업로드.

## 사용 순서
1. 런타임 → 런타임 유형 변경 → **GPU(T4)**.
2. 셀 2에서 경로 A면 `HF_TOKEN`·`HF_REPO` 입력, 경로 B면 비워 두고 파일 업로드.
3. 위에서부터 실행. 세션이 끊기면 다시 전체 실행 — 완료 타일은 건너뛴다
   (경로 A는 결과가 HF에 백업되므로 재개도 안전).
4. 끝나면 마지막 셀이 결과 CSV를 HF에 올리고(경로 A) 브라우저로도 내려준다.

In [ ]:
# 셀 1 — 환경 확인·설치
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 — 런타임 유형을 GPU로 바꿀 것')
%pip -q install "transformers>=4.45" huggingface_hub --upgrade


In [ ]:
# 셀 2 — 설정 + 타일 반입
HF_TOKEN = ''    # 경로 A: hf_... (read 권한이면 내려받기만, write면 결과 되올리기까지)
HF_REPO  = ''    # 경로 A: 예 'jyxnq/wp2026-tiles-pilot' (비공개 dataset)
SCENES   = []    # []이면 반입된 전 장면. 시험 예: ['C1_20260823015326_30092_00392940']

CLASSES = {
    'tower':      'electricity pylon',
    'substation': 'electrical substation',
}
TEXT = '. '.join(CLASSES.values()) + '.'
BOX_THRESHOLD, TEXT_THRESHOLD = 0.30, 0.25
SAVE_EVERY = 50

import glob
import os
import zipfile

BASE = '/content' if os.path.isdir('/content') else os.getcwd()   # Colab/Kaggle 겸용
TILES_DIR = os.path.join(BASE, 'tiles_pilot')
OUT_DIR = os.path.join(BASE, 'out')
os.makedirs(TILES_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

if HF_REPO:   # 경로 A — HF 비공개 데이터셋에서 반입
    from huggingface_hub import snapshot_download
    patterns = ['tiles_manifest.csv'] + ([f'{s}.zip' for s in SCENES] if SCENES else ['*.zip'])
    local = snapshot_download(repo_id=HF_REPO, repo_type='dataset',
                              token=HF_TOKEN or None, allow_patterns=patterns)
    import shutil
    for src in glob.glob(os.path.join(local, '*')):
        if src.endswith('.zip'):
            zipfile.ZipFile(src).extractall(TILES_DIR)
        elif os.path.isfile(src):
            shutil.copy(src, TILES_DIR)
else:         # 경로 B — 파일탭에 직접 올린 zip·매니페스트 사용
    import shutil
    for z in glob.glob(os.path.join(BASE, '*.zip')):
        zipfile.ZipFile(z).extractall(TILES_DIR)
    mf = os.path.join(BASE, 'tiles_manifest.csv')
    if os.path.isfile(mf):
        shutil.copy(mf, TILES_DIR)

assert os.path.isfile(os.path.join(TILES_DIR, 'tiles_manifest.csv')), \
    'tiles_manifest.csv 없음 — 경로 A는 HF_REPO 설정, 경로 B는 파일 업로드 확인'
print('장면 폴더:', len([d for d in os.listdir(TILES_DIR) if os.path.isdir(os.path.join(TILES_DIR, d))]))


In [ ]:
# 셀 3 — 매니페스트 로드 + 이어하기 준비 (HF에 이전 결과가 있으면 이어받음)
import csv

RAW = os.path.join(OUT_DIR, 'detections_raw.csv')
if HF_REPO and not os.path.isfile(RAW):
    try:
        from huggingface_hub import hf_hub_download
        prev = hf_hub_download(repo_id=HF_REPO, repo_type='dataset',
                               token=HF_TOKEN or None, filename='out/detections_raw.csv')
        os.system(f'cp "{prev}" "{RAW}"')
        print('HF의 이전 결과에서 이어서 진행')
    except Exception:
        pass

with open(os.path.join(TILES_DIR, 'tiles_manifest.csv'), encoding='utf-8') as f:
    manifest = [r for r in csv.DictReader(f)]
have = {d for d in os.listdir(TILES_DIR) if os.path.isdir(os.path.join(TILES_DIR, d))}
manifest = [r for r in manifest if r['scene'] in have]
if SCENES:
    manifest = [r for r in manifest if r['scene'] in SCENES]

done = set()
if os.path.isfile(RAW):
    with open(RAW, encoding='utf-8') as f:
        done = {r['tile'] for r in csv.DictReader(f)}
todo = [r for r in manifest if r['file'] not in done]
print(f'대상 타일 {len(manifest)} · 완료 {len(manifest)-len(todo)} · 남음 {len(todo)}')


In [ ]:
# 셀 4 — 모델 로드 (Grounding DINO base)
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

MODEL_ID = 'IDEA-Research/grounding-dino-base'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID).to(device).eval()

def label_to_class(label_text):
    """반환 라벨(프롬프트 부분 문자열)을 클래스 키로 환원."""
    for key, phrase in CLASSES.items():
        if any(w in label_text for w in phrase.split()):
            return key
    return 'other'


In [ ]:
# 셀 5 — 탐지 루프 (체크포인트: 로컬 플러시 + 1,000타일마다 HF 자동 백업)
import time
from PIL import Image

CKPT_EVERY = 1000   # N타일마다 HF에 중간 백업 (VM 교체 대비)

def hf_backup():
    if not (HF_REPO and HF_TOKEN):
        return
    try:
        from huggingface_hub import upload_file
        upload_file(path_or_fileobj=RAW, path_in_repo='out/detections_raw.csv',
                    repo_id=HF_REPO, repo_type='dataset', token=HF_TOKEN)
        print('  [백업] HF에 중간 결과 저장')
    except Exception as exc:
        print('  [백업 실패]', exc)

FIELDS = ['tile', 'scene', 'cls', 'score', 'px', 'py', 'x_5179', 'y_5179',
          'box_w_m', 'box_h_m']
new_file = not os.path.isfile(RAW)
fout = open(RAW, 'a', newline='', encoding='utf-8')
writer = csv.DictWriter(fout, fieldnames=FIELDS)
if new_file:
    writer.writeheader()

t0 = time.time()
for i, r in enumerate(todo):
    img = Image.open(os.path.join(TILES_DIR, r['file'])).convert('RGB')
    inputs = processor(images=img, text=TEXT, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    res = processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD,  # 구버전은 box_threshold=
        target_sizes=[img.size[::-1]])[0]
    a, b = float(r['a']), float(r['b'])
    d, e = float(r['d']), float(r['e'])
    x0, y0 = float(r['x0']), float(r['y0'])
    n_valid_w, n_valid_h = int(r['valid_w']), int(r['valid_h'])
    labels = res['text_labels'] if 'text_labels' in res else res['labels']
    for box, score, label in zip(res['boxes'], res['scores'], labels):
        bx0, by0, bx1, by1 = [float(v) for v in box]
        cx, cy = (bx0 + bx1) / 2, (by0 + by1) / 2
        if cx > n_valid_w or cy > n_valid_h:
            continue   # 패딩(검정) 영역의 허상
        writer.writerow({
            'tile': r['file'], 'scene': r['scene'],
            'cls': label_to_class(str(label)), 'score': round(float(score), 4),
            'px': round(cx, 1), 'py': round(cy, 1),
            'x_5179': round(x0 + a * cx + b * cy, 1),
            'y_5179': round(y0 + d * cx + e * cy, 1),
            'box_w_m': round((bx1 - bx0) * 0.5, 1),
            'box_h_m': round((by1 - by0) * 0.5, 1)})
    if (i + 1) % SAVE_EVERY == 0:
        fout.flush()
        rate = (i + 1) / (time.time() - t0)
        print(f'{i+1}/{len(todo)} 타일 · {rate:.1f}타일/초 · 남은 예상 {((len(todo)-i-1)/rate)/60:.0f}분')
    if (i + 1) % CKPT_EVERY == 0:
        fout.flush()
        hf_backup()
fout.close()
hf_backup()
print('탐지 완료 →', RAW)


In [ ]:
# 셀 6 — 중복 제거 (타일 겹침 128px로 같은 설비가 2번 잡히는 것 정리)
import math

RADIUS = {'tower': 25.0, 'substation': 80.0, 'other': 25.0}   # m

with open(RAW, encoding='utf-8') as f:
    rows = [r for r in csv.DictReader(f)]
rows.sort(key=lambda r: -float(r['score']))
kept = []
for r in rows:
    x, y, c = float(r['x_5179']), float(r['y_5179']), r['cls']
    if any(k['cls'] == c and math.hypot(float(k['x_5179']) - x, float(k['y_5179']) - y) < RADIUS[c]
           for k in kept):
        continue
    kept.append(r)

DEDUP = os.path.join(OUT_DIR, 'detections_dedup.csv')
with open(DEDUP, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(kept)

from collections import Counter
cnt = Counter((r['scene'][:33], r['cls']) for r in kept)
print(f'원시 {len(rows)} → 중복 제거 {len(kept)}')
for (scene, c), n in sorted(cnt.items()):
    print(f'  {scene}  {c:<11} {n}')


In [ ]:
# 셀 7 — 결과 반출: HF에 되올리기(경로 A, write 토큰 필요) + 브라우저 다운로드
if HF_REPO and HF_TOKEN:
    try:
        from huggingface_hub import upload_file
        for name in ('detections_raw.csv', 'detections_dedup.csv'):
            upload_file(path_or_fileobj=os.path.join(OUT_DIR, name),
                        path_in_repo=f'out/{name}', repo_id=HF_REPO,
                        repo_type='dataset', token=HF_TOKEN)
        print('HF 데이터셋 out/ 에 결과 업로드 완료 — 로컬에서 s3_hf_transfer.py download')
    except Exception as exc:
        print('HF 업로드 실패(read 전용 토큰?):', exc)
try:   # Colab에서만 — Kaggle 등 다른 환경에선 HF 반출로 충분
    from google.colab import files
    files.download(os.path.join(OUT_DIR, 'detections_dedup.csv'))
except ImportError:
    print('Colab 아님 — 결과는 HF 데이터셋과', OUT_DIR, '에 있음')


In [ ]:
# 셀 8 (선택) — 탐지 상위 몇 개를 눈으로 확인
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

N_SHOW = 6
best = sorted(kept, key=lambda r: -float(r['score']))[:N_SHOW]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, r in zip(axes.ravel(), best):
    img = Image.open(os.path.join(TILES_DIR, r['tile']))
    px, py = float(r['px']), float(r['py'])
    half_w = float(r['box_w_m']) / 0.5 / 2
    half_h = float(r['box_h_m']) / 0.5 / 2
    ax.imshow(img)
    ax.add_patch(patches.Rectangle((px - half_w, py - half_h), 2 * half_w, 2 * half_h,
                                   fill=False, color='red', lw=2))
    ax.set_title(f"{r['cls']} {float(r['score']):.2f}", fontsize=10)
    ax.axis('off')
plt.tight_layout(); plt.show()


## 다음 단계 (로컬에서)

1. 경로 A: Claude Code에서 `python scripts/s3_hf_transfer.py download` —
   결과 CSV가 `out/internal/` 로 내려온다(좌표 포함, 비공개 폴더).
   경로 B: 셀 7이 내려준 CSV를 `out/internal/` 에 넣는다.
2. "탐지 결과 대조해줘" — OSM 대조 기준과 매칭해 정밀도·재현율과 OSM 결측
   후보를 계산한다.
3. 문턱값 민감도 점검이 필요하면 셀 2의 BOX_THRESHOLD를 0.25/0.35로 바꿔
   재실행해 비교한다.